In [ ]:
# === TRAINING PRESETS ===
# Dataset: 220,000 samples available

PRESET = 1  # ✓ Standard preset for testing fixes

configs = {
    0: {  # Debug - 8k steps (~2 min) - fast iterations to test reward
        'TRAIN_DATA_SIZE': 16_384,
        'N_ENVS': 8,
        'N_STEPS': 256,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
    1: {  # Quick test - 32k steps (~5 min)
        'TRAIN_DATA_SIZE': 32_768,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 8,
    },
    2: {  # Standard - 131k steps (~20 min)
        'TRAIN_DATA_SIZE': 65_536,
        'N_ENVS': 8,
        'N_STEPS': 1024,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    3: {  # Extended - 524k steps (~1.5 hours)
        'TRAIN_DATA_SIZE': 131_072,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    4: {  # Max - 1.04M steps (~3 hours)
        'TRAIN_DATA_SIZE': 220_000,
        'N_ENVS': 8,
        'N_STEPS': 4096,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 32,
    },
}

# Load config
cfg = configs[PRESET]
TRAIN_DATA_SIZE = cfg['TRAIN_DATA_SIZE']
N_ENVS = cfg['N_ENVS']
N_STEPS = cfg['N_STEPS']
NUM_ITERATIONS = cfg['NUM_ITERATIONS']
BATCH_SIZE = cfg['BATCH_SIZE']

# Fixed params
LOOKBACK_WINDOW = 288
HIDDEN_DIM = 256
POLICY_LAYERS = [512, 256, 128]
VALUE_LAYERS = [256, 128]
LEARNING_RATE_START = 5e-4  # ✓ INCREASED from 3e-4 - faster learning from mistakes
LEARNING_RATE_DECAY = 0.0   # ✓ NO decay - keep it constant
N_EPOCHS = 10
ENT_COEF = 0.12             # ✓ High exploration
CLIP_RANGE = 0.3            # ✓ INCREASED from 0.2 - allow bigger policy updates
TARGET_KL  = 0.03           # ✓ INCREASED from 0.02 - less conservative updates

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_SAVE_PATH = "trading_bot"
VECNORM_SAVE_PATH = "vecnormalize.pkl"

# Calculated
STEPS_PER_ITERATION = N_ENVS * N_STEPS
TOTAL_TIMESTEPS = STEPS_PER_ITERATION * NUM_ITERATIONS

print(f"Preset {PRESET}: {TOTAL_TIMESTEPS:,} steps | {NUM_ITERATIONS} iters | {N_ENVS} envs | {TRAIN_DATA_SIZE:,} samples")

Preset 1: 163,840 steps | 8 iters | 10 envs | 32,768 samples


In [ ]:
import warnings
warnings.filterwarnings('ignore', message='enable_nested_tensor is True')

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import torch
import torch.nn as nn
import pandas as pd
import time
from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_enhanced_extractor import TradingEnhancedExtractor

# Load data
df = pd.read_pickle(DATA_PATH)
train_data = df.iloc[0:TRAIN_DATA_SIZE].reset_index(drop=True)
print(f"Loaded {len(df):,} rows | Training on {len(train_data):,} samples")

# Setup policy
policy_kwargs = dict(
    features_extractor_class=TradingEnhancedExtractor,
    features_extractor_kwargs=dict(hidden_dim=HIDDEN_DIM),
    net_arch=dict(pi=POLICY_LAYERS, vf=VALUE_LAYERS),
    activation_fn=torch.nn.GELU,
    ortho_init=False,
)

# Create environments
vec_env = make_vec_env(
    lambda: Monitor(SimpleTradingEnv(train_data, device="cuda", lookback_window=LOOKBACK_WINDOW)),
    n_envs=N_ENVS
)

# VecNormalize: balances features + clips observations to prevent extreme values
""" vec_env = VecNormalize(
    vec_env,
    norm_obs=True,         # Normalize observations (balances features)
    norm_reward=False,     # Keep raw rewards (already balanced: TP=50, SL=-25)
    clip_obs=10.0,         # Clip observations to [-10, 10] after normalization
    clip_reward=100.0,     # Clip rewards to [-100, 100] (allows TP=50, SL=-25, prevents extreme outliers)
) """

# Create model
model = PPO(
    "MultiInputPolicy",
    vec_env,
    device="cuda",
    learning_rate=lambda f: LEARNING_RATE_START * (1 - LEARNING_RATE_DECAY * f),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF,
    vf_coef=0.5,
    max_grad_norm=0.5,
    target_kl=TARGET_KL,
    stats_window_size=288,
    policy_kwargs=policy_kwargs,
    tensorboard_log="./tensorboard_logs/",
    verbose=2
)

# Reinitialize action network with smaller values for MultiDiscrete action space
# This prevents the model from being too confident in early steps
print("\n🔧 Reinitializing action network with smaller values for MultiDiscrete...")
for module in model.policy.action_net.modules():
    if isinstance(module, nn.Linear):
        nn.init.orthogonal_(module.weight, gain=0.5)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
print("✓ Action network reinitialized\n")

print(f"📊 Key Changes v23 - BALANCE-FOCUSED + REDUNDANT ACTION PENALTY:")
print(f"   ✓ REWARD v23: Balance growth + discrete action enforcement")
print(f"      - Redundant action penalty: -50 (CRITICAL FIX)")
print(f"         └─ Trying to LONG when already LONG → -50")
print(f"         └─ Trying to SHORT when already SHORT → -50")
print(f"      - Balance change: 0.5x multiplier (PRIMARY SIGNAL)")
print(f"      - Fast TP (<10 bars): 1.5x bonus (50-150 reward)")
print(f"      - Medium TP (11-50 bars): 1.2x bonus")
print(f"      - SL penalty: -15 to -30 (scales with loss)")
print(f"      - Exploration bonus: +2.0")
print(f"   ✓ CHANGES FROM v22:")
print(f"      - Added HARD penalty for redundant actions")
print(f"      - Forces agent to use HOLD or CLOSE when in position")
print(f"   ✓ GOAL: Maximize balance + learn proper discrete behavior\n")

try:
    # Train
    print(f"\nStarting training: {TOTAL_TIMESTEPS:,} steps...")
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
except KeyboardInterrupt:
    print("\n⏸ Training interrupted by user")
    
# Save
model.save(MODEL_SAVE_PATH)
print(f"\n✓ Saved: {MODEL_SAVE_PATH}")

# Save VecNormalize
#vec_env.save(VECNORM_SAVE_PATH)
#print(f"✓ Saved: {VECNORM_SAVE_PATH}")

# IMPORTANT: Collect expert trades from all environments
print("\n💾 Collecting expert trades from all environments...")
expert_trades_total = []

# Access environments correctly based on vec_env type
for i in range(vec_env.num_envs):
    # Try to get the unwrapped environment
    try:
        # For DummyVecEnv (single process): envs[i] is the Monitor wrapper
        env = vec_env.envs[i]
        # Unwrap Monitor to get SimpleTradingEnv
        while hasattr(env, 'env'):
            env = env.env
        
        if hasattr(env, 'expert_replay'):
            num_trades = len(env.expert_replay.expert_trades)
            if num_trades > 0:
                expert_trades_total.extend(env.expert_replay.expert_trades)
                print(f"  Env {i}: {num_trades} trades")
    except Exception as e:
        print(f"  Env {i}: Error accessing - {e}")

print(f"\n📊 Total trades collected: {len(expert_trades_total)}")

if expert_trades_total:
    # Deduplicate by (entry_step, pnl_percent) to avoid duplicates
    seen = set()
    unique_trades = []
    for trade in expert_trades_total:
        key = (trade['entry_step'], round(trade['pnl_percent'], 2))
        if key not in seen:
            seen.add(key)
            unique_trades.append(trade)

    # Save to disk using the replay system
    from environments.expert_trade_replay import ExpertTradeReplay
    replay_saver = ExpertTradeReplay()
    replay_saver.expert_trades = unique_trades
    replay_saver.save()
    
    # Print statistics
    stats = replay_saver.get_statistics()
    print(f"\n✓ Saved {stats['total_trades']} unique expert trades")
    print(f"  Avg PnL: {stats['avg_pnl']:.2f}%, Avg Duration: {stats['avg_duration']:.1f} steps")
    print(f"  Best: {stats['max_pnl']:.2f}% in {stats['min_duration']} steps")
else:
    print("\n⚠ No expert trades recorded this session")

print("\n✅ Training complete!")

Loaded 264,323 rows | Training on 32,768 samples
Using cuda device

🔧 Reinitializing action network with smaller values for MultiDiscrete...
✓ Action network reinitialized

📊 Key Changes v23 - BALANCE-FOCUSED + REDUNDANT ACTION PENALTY:
   ✓ REWARD v23: Balance growth + discrete action enforcement
      - Redundant action penalty: -50 (CRITICAL FIX)
         └─ Trying to LONG when already LONG → -50
         └─ Trying to SHORT when already SHORT → -50
      - Balance change: 0.5x multiplier (PRIMARY SIGNAL)
      - Fast TP (<10 bars): 1.5x bonus (50-150 reward)
      - Medium TP (11-50 bars): 1.2x bonus
      - SL penalty: -15 to -30 (scales with loss)
      - Exploration bonus: +2.0
   ✓ CHANGES FROM v22:
      - Added HARD penalty for redundant actions
      - Forces agent to use HOLD or CLOSE when in position
   ✓ GOAL: Maximize balance + learn proper discrete behavior


Starting training: 163,840 steps...
Logging to ./tensorboard_logs/PPO_314


Output()